In [1]:
import pandas as pd 
import numpy as np
import torch
from torch_geometric.data import Data
import torch.nn.functional as F
import warnings
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
import torch.nn as nn
from gensim.models import Word2Vec
from tqdm import tqdm
import torch.nn.functional as F
import math
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, roc_curve
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

/home/cs/grad/nasrm1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/usr/lib/python3/dist-packages/paramiko/transport.py:261: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


In [2]:
def prepare_graph(df):
    
    def process_node(node, action, node_dict, label_dict, dummies, node_type):
        node_dict.setdefault(node, []).append(action)
        label_dict[node] = dummies.get(getattr(row, node_type), -1)  

    nodes = {}
    labels = {}
    edges = []
    dummies = {
        "7998762093665332071": 0, "14709879154498484854": 1, "10991425273196493354": 2,
        "14871526952859113360": 3, "8771628573506871447": 4, "7877121489144997480": 5,
        "17841021884467483934": 6, "7895447931126725167": 7, "15125250455093594050": 8,
        "8664433583651064836": 9, "14377490526132269506": 10, "15554536683409451879": 11,
        "8204541918505434145": 12, "14356114695140920775": 13
    }

    for row in df.itertuples():
        process_node(row.actorID, row.action, nodes, labels, dummies, 'actor_type')
        process_node(row.objectID, row.action, nodes, labels, dummies, 'object')
        edges.append((row.actorID, row.objectID))

    features = [nodes[node] for node in nodes]
    feat_labels = [labels[node] for node in nodes]
    edge_index = [[], []]
    for src, dst in edges:
        src_index = list(nodes.keys()).index(src)
        dst_index = list(nodes.keys()).index(dst)
        edge_index[0].append(src_index)
        edge_index[1].append(dst_index)

    return features, feat_labels, edge_index, list(nodes.keys())


In [3]:
class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, 20, normalize=True)
        self.linear = nn.Linear(in_features=20,out_features=out_channel)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.linear(x)
        return F.softmax(x, dim=1)

In [4]:
class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]

def infer(document):
    word_embeddings = [w2vmodel.wv[word] for word in document if word in w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(20)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(30)
w2vmodel = Word2Vec.load("trained_weights/unicorn/unicorn.model")
model = GCN(30,14).to(device)


In [9]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import random

number_of_nodes = []
cross_label_similarities = []

for i in tqdm(range(125, 150), desc="Processing Graphs"):
    with open(f"unicorn/{i}.txt") as f:
        data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame(data, columns=['actorID', 'actor_type', 'objectID', 'object', 'action', 'timestamp'])
    df.sort_values(by='timestamp', ascending=True, inplace=True)
    df = df.dropna()
    
    phrases, labels, edges, mapp = prepare_graph(df)

    number_of_nodes.append(len(phrases))

    # Get indices for each label
    zero_indices = [idx for idx, lbl in enumerate(labels) if lbl == 0]
    one_indices  = [idx for idx, lbl in enumerate(labels) if lbl == 1]

    # Need at least 1 node from each label to compute cross-label similarity
    if zero_indices and one_indices:
        # Infer embeddings
        zero_nodes = np.array([infer(phrases[idx]) for idx in zero_indices])
        one_nodes  = np.array([infer(phrases[idx]) for idx in one_indices])

        # Sample up to 1000 from each label
        if len(zero_nodes) > 100:
            zero_nodes = zero_nodes[random.sample(range(len(zero_nodes)), 100)]
        if len(one_nodes) > 100:
            one_nodes = one_nodes[random.sample(range(len(one_nodes)), 100)]

        # Compute pairwise cosine similarity between label 0 and label 1 nodes
        sim_matrix = cosine_similarity(zero_nodes, one_nodes)
        avg_cross_sim = np.mean(sim_matrix)
        cross_label_similarities.append(avg_cross_sim)

print("Average number of nodes per graph:", np.mean(number_of_nodes))
print("Average cosine similarity between label=0 and label=1 nodes:", np.mean(cross_label_similarities))


Processing Graphs: 100%|██████████| 25/25 [02:56<00:00,  7.07s/it]

Average number of nodes per graph: 13569.12
Average cosine similarity between label=0 and label=1 nodes: 0.17311196


# Before Evasion

In [ ]:
thresh = 330

correct_attack = 0

for i in tqdm(range(125, 150), desc="Processing Graphs"):
    f = open(f"unicorn/{i}.txt")
    data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame(data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df.sort_values(by='timestamp', ascending=True,inplace=True)
    df = df.dropna()
    
    phrases,labels,edges,mapp = prepare_graph(df)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)  
    
    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device), 
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )
    
    graph.n_id = torch.arange(graph.num_nodes, device=device)

    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool, device=device)

    for m_n in range(20):
        model.load_state_dict(torch.load(f'trained_weights/unicorn/unicorn{m_n}.pth'))
        model.eval()
        out = model(graph.x, graph.edge_index)

        sorted, indices = out.sort(dim=1,descending=True)
        conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
        conf = (conf - conf.min()) / conf.max()

        pred = indices[:,0]
        cond = (pred == graph.y)
        flag[graph.n_id[cond]] = torch.logical_and(
        flag[graph.n_id[cond]], 
        torch.tensor([False]*len(flag[graph.n_id[cond]]), dtype=torch.bool, device=device)
        )

    if  flag.sum().item() > thresh:
        correct_attack = correct_attack + 1
   
    # print(flag.sum().item(), (flag.sum().item() / len(flag))*100)
print("Number of correct attack predictions: ", correct_attack)

Processing Graphs:   0%|          | 0/25 [00:05<?, ?it/s]

                    actorID            actor_type              objectID  \
57     13092160573356321650   7877121489144997480   2005817169390137492   
91       611441087162399973   7998762093665332071  10038474408652424193   
51     15902583369731029962   7877121489144997480    227722660868761057   
1695    6642881403180538625  10991425273196493354  14807344685136098291   
11749   8730267468155635675   7877121489144997480  12315976371151318521   

                     object                action timestamp  
57     14709879154498484854  16294740304317166071         1  
91     14709879154498484854  17885712477553841827        10  
51      7998762093665332071  13451921668877089297       100  
1695   10991425273196493354   8852477614386305826      1000  
11749  14709879154498484854  16294740304317166071     10000  
Number of correct attack predictions:  0


In [21]:
correct_benign = 0

for i in tqdm(range(100,125), desc="Processing Graphs"):
    # print(f"Graph #: {i}")
    f = open(f"unicorn/{i}.txt")
    data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df.sort_values(by='timestamp', ascending=True,inplace=True)
    df = df.dropna()

    phrases,labels,edges,mapp = prepare_graph(df)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)  

    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device), 
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )
    
    graph.n_id = torch.arange(graph.num_nodes, device=device)

    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool, device=device)

    for m_n in range(20):
        model.load_state_dict(torch.load(f'trained_weights/unicorn/unicorn{m_n}.pth'))
        model.eval()
        out = model(graph.x, graph.edge_index)

        sorted, indices = out.sort(dim=1,descending=True)
        conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
        conf = (conf - conf.min()) / conf.max()

        pred = indices[:,0]
        cond = (pred == graph.y)
        flag[graph.n_id[cond]] = torch.logical_and(
        flag[graph.n_id[cond]], 
        torch.tensor([False]*len(flag[graph.n_id[cond]]), dtype=torch.bool, device=device)
        )

    if flag.sum().item() <= thresh:
        correct_benign = correct_benign + 1
            
    # print(flag.sum().item(), (flag.sum().item() / len(flag))*100)

print("Number of correct benign predictions: ", correct_benign)

Processing Graphs: 100%|██████████| 25/25 [04:02<00:00,  9.69s/it]

Number of correct benign predictions:  24


In [22]:
TP = correct_attack
FP = 25 - correct_benign
TN = correct_benign
FN = 25 - correct_attack

FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
TNR = TN / (TN + FP) if (TN + FP) > 0 else 0

print(f"Number of True Positives (TP): {TP}")
print(f"Number of False Positives (FP): {FP}")
print(f"Number of False Negatives (FN): {FN}")
print(f"Number of True Negatives (TN): {TN}\n")

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TPR  
print(f"Precision: {precision}")
print(f"Recall: {recall}")

fscore = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"Fscore: {fscore}\n")

print("False Positive Rate (FPR):", round(FPR, 2))
print("True Positive Rate (TPR):", round(TPR, 2))
print("True Negative Rate (TNR):", round(TNR, 2))


Number of True Positives (TP): 24
Number of False Positives (FP): 1
Number of False Negatives (FN): 1
Number of True Negatives (TN): 24

Precision: 0.96
Recall: 0.96
Fscore: 0.96

False Positive Rate (FPR): 0.04
True Positive Rate (TPR): 0.96
True Negative Rate (TNR): 0.96


# Functions I Use For Evasion

In [5]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

def split_nodes_by_label_exclude_malicious(phrases, labels, indices_of_malicious_nodes):
    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  # Convert to set for faster lookup

    for node_idx, node_features in enumerate(phrases):
        # Check if the node index is in the malicious nodes list
        if node_idx not in malicious_set:
            node_label = labels[node_idx]
            node_groups[node_label].append(node_idx)

    return dict(node_groups)

def filter_nodes_by_phrase_length(node_groups, phrases, min_len=20, max_len=40):

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        filtered_groups[label] = [
            idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len
        ]

    return filtered_groups

def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):
    top_nodes_by_similarity = {}

    for label in malicious_groups.keys():
        if label in filtered_nodes_grouped:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)  # Shape: (filtered, malicious)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)  # Max similarity per filtered node

            # Step 1: Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Step 2: Select most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices 

    return top_nodes_by_similarity


def duplicate_and_replace(df, target_id, new_id):
    # Find rows where target_id appears in actorID or objectID
    mask = df['actorID'].isin([target_id]) | df['objectID'].isin([target_id])
    selected_rows = df[mask]

    # Replace target_id with new_id in actorID and objectID using replace() method
    selected_rows[['actorID', 'objectID']] = selected_rows[['actorID', 'objectID']].replace(target_id, new_id)

    # Return the modified rows, no need to append to df within the function
    return selected_rows

# After Evasion

In [ ]:
import os
def df_to_evasion_txt(df, i):
    out_file = f"unicorn/evasion/{i}.txt"
    # select & order the columns, then dump
    df[['actorID','actor_type','objectID','object','action','timestamp']] \
      .to_csv(out_file, sep='\t', header=False, index=False) 

In [ ]:
# It takes 30 minutes to run this cell
thresh = 330

correct_attack = 0

number_of_triggred_actions = []

number_of_flagged_nodes = []

error_percentage = []

for i in tqdm(range(125, 150), desc="Processing Graphs"):

    graph_number = i

    flagged_indices = []
    flagged_labels = []
    # print(f"Graph #: {i}")
    f = open(f"unicorn/{i}.txt")
    data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df.sort_values(by='timestamp', ascending=True,inplace=True)
    df = df.dropna()
    
    phrases,labels,edges,mapp = prepare_graph(df)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)  

    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device), 
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )
    
    graph.n_id = torch.arange(graph.num_nodes, device=device)

    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool, device=device)

    for m_n in range(20):
        model.load_state_dict(torch.load(f'trained_weights/unicorn/unicorn{m_n}.pth'))
        model.eval()
        out = model(graph.x, graph.edge_index)

        sorted, indices = out.sort(dim=1,descending=True)
        conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
        conf = (conf - conf.min()) / conf.max()

        pred = indices[:,0]

        cond = (pred == graph.y)

        flag[graph.n_id[cond]] = torch.logical_and(
        flag[graph.n_id[cond]], 
        torch.tensor([False]*len(flag[graph.n_id[cond]]), dtype=torch.bool, device=device)
        )

    # Append indices of flagged nodes (those with True values) to flagged_indices list
    flagged_indices.extend(graph.n_id[flag].cpu().numpy())

    # Extract corresponding labels for flagged indices
    flagged_labels.extend(graph.y[flag].cpu().numpy())

    # Count total nodes per label
    unique_labels, label_counts = np.unique(labels, return_counts=True)
    label_dict = dict(zip(unique_labels, label_counts))

    # Count flagged nodes per label
    unique_flagged_labels, flagged_counts = np.unique(flagged_labels, return_counts=True)
    flagged_dict = dict(zip(unique_flagged_labels, flagged_counts))

    ############################################################################################################
    # EVASION
    ############################################################################################################

    # Step 1: Benign Node Selection
    malicious_groups = split_malicious_nodes_by_label(flagged_indices, labels) 

    # Split all nodes by their labels excluding the malicious nodes
    all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(phrases, labels, flagged_indices)

    # Step 2: Minimal Interaction Selection
    filtered_nodes_grouped = filter_nodes_by_phrase_length(all_nodes_grouped_excluding_malicious, phrases)

    # Step 3: Contextual Consistency Filtering, Compute top similar nodes by similarity
    top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes)

    # Prepare a list to collect the modified rows
    modified_rows = []

    # Precompute all malicious node IDs and their most similar benign node IDs in advance
    malicious_and_benign = {}

    for i in flagged_indices:
        malicious_node_ID = mapp[i]  # Malicious node ID from the mapping
        malicious_label = labels[i]  # Get the malicious node label
        
        try:
            # Get all similar benign node IDs for the malicious label, limit to 10
            benign_candidates = top_nodes_by_similarity[malicious_label]
            # benign_candidates = filtered_nodes_grouped[malicious_label]
            if len(benign_candidates) > 25:
                benign_candidates = benign_candidates[:25]  # Limit to top 25

            similar_benign_node_IDs = [mapp[node] for node in benign_candidates]
        except KeyError:
            continue
        
        # Store the mapping of malicious node to all similar benign node IDs
        malicious_and_benign[malicious_node_ID] = similar_benign_node_IDs

    # Use tqdm for the longest loop
    for malicious_node_ID, benign_node_IDs in malicious_and_benign.items():
        for benign_node_ID in benign_node_IDs:  # Iterate over all similar benign nodes (max 10)
            selected_rows = duplicate_and_replace(df, benign_node_ID, malicious_node_ID)
            modified_rows.append(selected_rows)

    # After all iterations, concatenate all the new rows once
    df_curated = pd.concat([df] + modified_rows, ignore_index=True)

    # df_to_evasion_txt(df_curated, graph_number)

    number_of_triggred_actions.append(len(modified_rows))

    phrases,labels,edges,mapp = prepare_graph(df_curated)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)  

    graph = Data(
        x=torch.tensor(nodes, dtype=torch.float).to(device),
        y=torch.tensor(labels, dtype=torch.long).to(device), 
        edge_index=torch.tensor(edges, dtype=torch.long).to(device)
    )
    
    graph.n_id = torch.arange(graph.num_nodes, device=device)

    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool, device=device)

    for m_n in range(20):
        model.load_state_dict(torch.load(f'trained_weights/unicorn/unicorn{m_n}.pth'))
        model.eval()
        out = model(graph.x, graph.edge_index)

        sorted, indices = out.sort(dim=1,descending=True)
        conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
        conf = (conf - conf.min()) / conf.max()

        pred = indices[:,0]

        cond = (pred == graph.y)

        flag[graph.n_id[cond]] = torch.logical_and(
        flag[graph.n_id[cond]], 
        torch.tensor([False]*len(flag[graph.n_id[cond]]), dtype=torch.bool, device=device)
        )

    if  flag.sum().item() > thresh:
        correct_attack = correct_attack + 1

    number_of_flagged_nodes.append(flag.sum().item())

    error_percentage.append((flag.sum().item() / len(flag))*100)

print("Number of correct attack predictions: ", correct_attack)
   

Processing Graphs: 100%|██████████| 25/25 [17:33<00:00, 42.12s/it]

Number of correct attack predictions:  0


In [8]:
for i, count in enumerate(number_of_triggred_actions):
    print(f"Graph {i}: {count} triggered actions")

Graph 0: 5683 triggered actions
Graph 1: 5645 triggered actions
Graph 2: 3341 triggered actions
Graph 3: 5569 triggered actions
Graph 4: 5688 triggered actions
Graph 5: 5650 triggered actions
Graph 6: 5722 triggered actions
Graph 7: 5760 triggered actions
Graph 8: 5670 triggered actions
Graph 9: 5630 triggered actions
Graph 10: 5651 triggered actions
Graph 11: 5716 triggered actions
Graph 12: 5594 triggered actions
Graph 13: 5710 triggered actions
Graph 14: 127 triggered actions
Graph 15: 5672 triggered actions
Graph 16: 5673 triggered actions
Graph 17: 5734 triggered actions
Graph 18: 5640 triggered actions
Graph 19: 5639 triggered actions
Graph 20: 5645 triggered actions
Graph 21: 5726 triggered actions
Graph 22: 5681 triggered actions
Graph 23: 5584 triggered actions
Graph 24: 5603 triggered actions


In [9]:
for i, count in enumerate(number_of_flagged_nodes):
    print(f"Graph {i}: {count} flagged nodes")

Graph 0: 207 flagged nodes
Graph 1: 197 flagged nodes
Graph 2: 248 flagged nodes
Graph 3: 169 flagged nodes
Graph 4: 228 flagged nodes
Graph 5: 273 flagged nodes
Graph 6: 287 flagged nodes
Graph 7: 262 flagged nodes
Graph 8: 234 flagged nodes
Graph 9: 259 flagged nodes
Graph 10: 240 flagged nodes
Graph 11: 240 flagged nodes
Graph 12: 202 flagged nodes
Graph 13: 244 flagged nodes
Graph 14: 236 flagged nodes
Graph 15: 299 flagged nodes
Graph 16: 234 flagged nodes
Graph 17: 213 flagged nodes
Graph 18: 223 flagged nodes
Graph 19: 287 flagged nodes
Graph 20: 225 flagged nodes
Graph 21: 279 flagged nodes
Graph 22: 270 flagged nodes
Graph 23: 245 flagged nodes
Graph 24: 162 flagged nodes


In [27]:
thresh = 330

correct_benign = 0

for i in tqdm(range(100,125), desc="Processing Graphs"):
    # print(f"Graph #: {i}")
    f = open(f"unicorn/{i}.txt")
    data = f.read().split('\n')

    data = [line.split('\t') for line in data]
    df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df.sort_values(by='timestamp', ascending=True,inplace=True)
    df = df.dropna()

    phrases,labels,edges,mapp = prepare_graph(df)

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)  

    graph = Data(
        x=torch.tensor(nodes,dtype=torch.float).to(device),
        y=torch.tensor(labels,dtype=torch.long).to(device), 
        edge_index=torch.tensor(edges,dtype=torch.long).to(device)
    )
    
    graph.n_id = torch.arange(graph.num_nodes, device=device)

    flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool, device=device)

    for m_n in range(20):
        model.load_state_dict(torch.load(f'trained_weights/unicorn/unicorn{m_n}.pth'))
        model.eval()
        out = model(graph.x, graph.edge_index)

        sorted, indices = out.sort(dim=1,descending=True)
        conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
        conf = (conf - conf.min()) / conf.max()

        pred = indices[:,0]

        cond = (pred == graph.y)

        flag[graph.n_id[cond]] = torch.logical_and(
        flag[graph.n_id[cond]], 
        torch.tensor([False]*len(flag[graph.n_id[cond]]), dtype=torch.bool, device=device)
        )

    if flag.sum().item() <= thresh:
        correct_benign = correct_benign + 1
            
    # print(flag.sum().item(), (flag.sum().item() / len(flag))*100)
print("Number of correct benign predictions: ", correct_benign)

Processing Graphs:   0%|          | 0/25 [00:00<?, ?it/s]

Processing Graphs: 100%|██████████| 25/25 [04:02<00:00,  9.72s/it]

Number of correct benign predictions:  24


In [19]:
correct_benign = 24

TP = correct_attack
FP = 25 - correct_benign
TN = correct_benign
FN = 25 - correct_attack

FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
TNR = TN / (TN + FP) if (TN + FP) > 0 else 0

print(f"Number of True Positives (TP): {TP}")
print(f"Number of False Positives (FP): {FP}")
print(f"Number of False Negatives (FN): {FN}")
print(f"Number of True Negatives (TN): {TN}\n")

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TPR  
print(f"Precision: {precision}")
print(f"Recall: {recall}")

fscore = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print(f"Fscore: {fscore}\n")

print("False Positive Rate (FPR):", round(FPR, 2))
print("True Positive Rate (TPR):", round(TPR, 2))
print("True Negative Rate (TNR):", round(TNR, 2))



Number of True Positives (TP): 0
Number of False Positives (FP): 1
Number of False Negatives (FN): 25
Number of True Negatives (TN): 24

Precision: 0.0
Recall: 0.0
Fscore: 0

False Positive Rate (FPR): 0.04
True Positive Rate (TPR): 0.0
True Negative Rate (TNR): 0.96
